In [15]:
from google.colab import drive
drive.mount('/content/gdrive')

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


In [16]:
!pip install -q transformers

In [17]:
import pandas as pd
import numpy as np
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch.nn as nn

In [18]:
df = pd.read_csv('/content/gdrive/MyDrive/Colab Notebooks/course/datasets/data_coindesk_base.csv')
df = df[['BODY', 'PUBLISHED_ON']]
df.dropna(inplace=True)
# сразу сокращаем длины статей до необходимого размера
df['BODY'] = df['BODY'].apply(lambda x: x[:512] if len(x) > 512 else x)


In [19]:
model = AutoModelForSequenceClassification.from_pretrained('ProsusAI/finbert')
tokenizer = AutoTokenizer.from_pretrained('ProsusAI/finbert')

finbert_model_path = '/content/gdrive/MyDrive/Colab Notebooks/course/notebooks/transformers/model_finbert_v0.zip'
loaded_state_dict = torch.load(finbert_model_path, map_location=torch.device('cpu'))

model.load_state_dict(loaded_state_dict)

<All keys matched successfully>

In [36]:
model.config.id2label

{0: 'positive', 1: 'negative', 2: 'neutral'}

In [46]:
def sentiment(text, model=model, tokenizer=tokenizer):

    toknes = tokenizer(text, truncation=True,
                       padding='max_length', max_length=512, padding_side='right')

    # also add batch dimension
    input_ids = torch.tensor(toknes['input_ids']).unsqueeze(0)
    attention_mask = torch.tensor(toknes['attention_mask']).unsqueeze(0)
    with torch.no_grad():
        print('start evaluation sentiment...')
        logits = model(input_ids, attention_mask=attention_mask).logits
        sm = torch.nn.Softmax(dim=-1)
        probs = sm(logits)

        outputs = probs[0].argmax(dim=-1)

    return round(outputs.item(), 3), probs[0]


In [47]:
sentiment(df['BODY'][1])

start evaluation sentiment...


(2, tensor([0.4408, 0.0603, 0.4989]))

In [43]:
df['BODY'][1]

'XRP’s recent drop to around $2.29 stems from Bitcoin’s broader market selloff, not inherent weaknesses in the XRP ecosystem. Despite the dip, institutional investors are accumulating over 149 million XRP'

Недавнее падение XRP до уровня около 2,29 доллара связано с более широкой распродажей биткоина на рынке, а не с присущими экосистеме XRP недостатками. Несмотря на падение, институциональные инвесторы накапливают более 149 миллионов XRP.